# 📊 M365 Agent Templates — GitHub Metrics Dashboard

Pulls release downloads, traffic stats, and real-time view badges from GitHub.

**Data Sources:**
- GitHub REST API (releases, traffic views/clones, popular content, referrers)
- hits.sh badges (real-time view counts per folder README)

**Schedule:** Run this notebook on a schedule (e.g., daily) to build historical trend data.

In [ ]:
# ── Configuration ──
OWNER = "microsoft"
REPO = "m365-agent-templates"
# Set your GitHub PAT as a Fabric secret or paste here (repo scope needed for traffic)
GITHUB_TOKEN = ""  # TODO: Replace with your PAT or use mssparkutils.credentials

# Folders with README tracking badges
TRACKED_PATHS = [
    "",                   # root
    "Plan My Day",
    "Executive Briefing",
    "Know My Customer",
    "My Company Policy",
    "Request Tracker",
]

BASE_URL = f"https://api.github.com/repos/{OWNER}/{REPO}"
REPO_URL = f"https://github.com/{OWNER}/{REPO}"

In [ ]:
import requests
import re
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import *

headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": "2022-11-28",
}

collected_at = datetime.utcnow().isoformat()
print(f"📊 Collecting metrics at {collected_at}")

## 📦 Release Asset Downloads

In [ ]:
# ── Fetch all releases with pagination ──
all_releases = []
page = 1
while True:
    resp = requests.get(f"{BASE_URL}/releases", headers=headers, params={"per_page": 100, "page": page})
    resp.raise_for_status()
    data = resp.json()
    if not data:
        break
    all_releases.extend(data)
    page += 1

# ── Build rows ──
download_rows = []
for release in all_releases:
    for asset in release.get("assets", []):
        download_rows.append(Row(
            collected_at=collected_at,
            release_name=release["name"] or release["tag_name"],
            tag=release["tag_name"],
            published_at=release["published_at"],
            prerelease=release["prerelease"],
            asset_name=asset["name"],
            download_count=asset["download_count"],
            size_mb=round(asset["size"] / (1024 * 1024), 2),
        ))

if download_rows:
    df_downloads = spark.createDataFrame(download_rows)
    df_downloads.write.mode("append").format("delta").saveAsTable("release_downloads")
    print(f"✅ Saved {len(download_rows)} release asset records")
    display(df_downloads)
else:
    print("ℹ️ No release assets found")

## 👁️ Traffic — Page Views (last 14 days)

In [ ]:
resp = requests.get(f"{BASE_URL}/traffic/views", headers=headers)
if resp.status_code == 200:
    views_data = resp.json()
    view_rows = []
    for v in views_data.get("views", []):
        view_rows.append(Row(
            collected_at=collected_at,
            date=v["timestamp"][:10],
            views=v["count"],
            unique_visitors=v["uniques"],
        ))
    if view_rows:
        df_views = spark.createDataFrame(view_rows)
        df_views.write.mode("append").format("delta").saveAsTable("traffic_views")
        print(f"✅ Saved {len(view_rows)} daily view records (total: {views_data['count']}, unique: {views_data['uniques']})")
        display(df_views)
else:
    print(f"⚠️ Traffic views: {resp.status_code} — need push access token")

## 📋 Traffic — Clones (last 14 days)

In [ ]:
resp = requests.get(f"{BASE_URL}/traffic/clones", headers=headers)
if resp.status_code == 200:
    clones_data = resp.json()
    clone_rows = []
    for c in clones_data.get("clones", []):
        clone_rows.append(Row(
            collected_at=collected_at,
            date=c["timestamp"][:10],
            clones=c["count"],
            unique_cloners=c["uniques"],
        ))
    if clone_rows:
        df_clones = spark.createDataFrame(clone_rows)
        df_clones.write.mode("append").format("delta").saveAsTable("traffic_clones")
        print(f"✅ Saved {len(clone_rows)} daily clone records (total: {clones_data['count']}, unique: {clones_data['uniques']})")
        display(df_clones)
else:
    print(f"⚠️ Traffic clones: {resp.status_code} — need push access token")

## 🔥 Popular Content & Referrers

In [ ]:
# ── Popular paths ──
resp = requests.get(f"{BASE_URL}/traffic/popular/paths", headers=headers)
if resp.status_code == 200:
    path_rows = [Row(collected_at=collected_at, path=p["path"], views=p["count"], unique_visitors=p["uniques"]) for p in resp.json()]
    if path_rows:
        df_paths = spark.createDataFrame(path_rows)
        df_paths.write.mode("append").format("delta").saveAsTable("popular_content")
        print(f"✅ Saved {len(path_rows)} popular content records")
        display(df_paths)
else:
    print(f"⚠️ Popular content: {resp.status_code}")

# ── Referrers ──
resp = requests.get(f"{BASE_URL}/traffic/popular/referrers", headers=headers)
if resp.status_code == 200:
    ref_rows = [Row(collected_at=collected_at, referrer=r["referrer"], views=r["count"], unique_visitors=r["uniques"]) for r in resp.json()]
    if ref_rows:
        df_refs = spark.createDataFrame(ref_rows)
        df_refs.write.mode("append").format("delta").saveAsTable("referrers")
        print(f"✅ Saved {len(ref_rows)} referrer records")
        display(df_refs)
else:
    print(f"⚠️ Referrers: {resp.status_code}")

## 🎯 Real-Time README View Counts (hits.sh)

In [ ]:
# ── Parse view counts from hits.sh SVG badges ──
badge_rows = []
for folder in TRACKED_PATHS:
    if folder == "":
        tracking_url = f"{REPO_URL}"
        label = "/ (root)"
    else:
        tracking_url = f"{REPO_URL}/tree/main/{folder}"
        label = f"/{folder}/"

    svg_url = f"https://hits.sh/{tracking_url}.svg?label=views&color=0078D4"
    try:
        resp = requests.get(svg_url, timeout=10)
        # Extract count from aria-label="views: N"
        match = re.search(r'aria-label="views:\s*(\d+)"', resp.text)
        count = int(match.group(1)) if match else 0
        badge_rows.append(Row(
            collected_at=collected_at,
            folder=label,
            tracking_url=tracking_url,
            view_count=count,
        ))
        print(f"  {label} → {count} views")
    except Exception as e:
        print(f"  ⚠️ {label}: {e}")

if badge_rows:
    df_badges = spark.createDataFrame(badge_rows)
    df_badges.write.mode("append").format("delta").saveAsTable("readme_views")
    print(f"\n✅ Saved {len(badge_rows)} README view count records")
    display(df_badges)

## 📈 Summary Dashboard View

In [ ]:
# ── Quick summary of latest data ──
print("\n" + "═" * 60)
print("  📊 M365 AGENT TEMPLATES — METRICS SUMMARY")
print("═" * 60)

# Release downloads
try:
    df = spark.sql("SELECT SUM(download_count) as total FROM release_downloads WHERE collected_at = (SELECT MAX(collected_at) FROM release_downloads)")
    total = df.collect()[0]["total"] or 0
    print(f"\n  📦 Total Release Downloads: {total}")
except:
    print("\n  📦 Release Downloads: No data yet")

# Traffic views
try:
    df = spark.sql("SELECT SUM(views) as total, SUM(unique_visitors) as uniq FROM traffic_views WHERE collected_at = (SELECT MAX(collected_at) FROM traffic_views)")
    row = df.collect()[0]
    print(f"  👁️ Traffic Views (14d): {row['total'] or 0} views, {row['uniq'] or 0} unique")
except:
    print("  👁️ Traffic Views: No data yet")

# Clones
try:
    df = spark.sql("SELECT SUM(clones) as total, SUM(unique_cloners) as uniq FROM traffic_clones WHERE collected_at = (SELECT MAX(collected_at) FROM traffic_clones)")
    row = df.collect()[0]
    print(f"  📋 Clones (14d): {row['total'] or 0} clones, {row['uniq'] or 0} unique")
except:
    print("  📋 Clones: No data yet")

# README views
try:
    df = spark.sql("SELECT folder, view_count FROM readme_views WHERE collected_at = (SELECT MAX(collected_at) FROM readme_views) ORDER BY view_count DESC")
    print("  🎯 README Views (real-time):")
    for row in df.collect():
        print(f"     {row['folder']} → {row['view_count']} views")
except:
    print("  🎯 README Views: No data yet")

print("\n" + "═" * 60)
print(f"  ✅ Collection complete: {collected_at}")
print("═" * 60)